# Company Policy RAG — Exploration & Evaluation

Use this notebook to:
1. Inspect indexed chunks and metadata
2. Run retrieval-only queries (no generation)
3. Prototype evaluation metrics before productionizing

**Prerequisites:** Run `python scripts/index_documents.py` first.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import settings
from src.indexing import load_index, create_retriever, index_exists
from src.utils import format_citations

print(f"Project root: {PROJECT_ROOT}")
print(f"Index exists: {index_exists()}")
print(f"Chunk size: {settings.chunk_size} | top_k: {settings.similarity_top_k}")

In [ ]:
# Load index and inspect sample chunk metadata
if not index_exists():
    raise RuntimeError("Run: python scripts/index_documents.py")

index = load_index()
retriever = create_retriever(index)

# Peek at first few nodes in the docstore
docstore = index.docstore.docs
print(f"Total nodes in index: {len(docstore)}")

for i, (node_id, node) in enumerate(list(docstore.items())[:3]):
    print(f"\n--- Node {i+1} ---")
    print(f"Metadata: {node.metadata}")
    print(f"Text preview: {(node.text or '')[:200]}...")

In [ ]:
# Retrieval-only evaluation — tune queries for your golden set
TEST_QUERIES = [
    "What is the PTO policy?",
    "How does remote work work?",
    "What are confidentiality obligations?",
]

for query in TEST_QUERIES:
    print(f"\n{'='*60}")  
    print(f"Query: {query}")
    nodes = retriever.retrieve(query)
    citations = format_citations(nodes)
    for c in citations[:3]:
        print(f"  [{c['score']}] {c['source_file']} p.{c['page_number']} — {c['section_title']}")
        print(f"       {c['excerpt'][:120]}...")

In [ ]:
# Full RAG query via query engine (generation + retrieval)
from src.indexing import create_query_engine

query_engine = create_query_engine(index)
response = query_engine.query("Summarize the leave policy.")
print(response)

## Next: Evaluation Harness

Build a golden set of `(query, expected_answer, supporting_pages)` tuples and integrate **RAGAS**:

```bash
pip install ragas
```

Track `context_precision`, `context_recall`, and `faithfulness` on every indexing change.